# Developer Flow & Interruption Data Cleaning
  


## Step 1: Import the libraries we need

In [1]:
import pandas as pd
import numpy as np

## Step 2: Load all 6 files

In [2]:
fact_log = pd.read_csv("fact_developer_activity_log.csv")

In [3]:
fact_daily = pd.read_csv("fact_flow_daily.csv")

In [4]:
dim_developer = pd.read_csv("dim_developer.csv")

In [5]:
dim_activity = pd.read_csv("dim_activity_type.csv")

In [6]:
dim_interruption = pd.read_csv("dim_interruption.csv")

In [7]:
dim_date = pd.read_csv("dim_date.csv")

Let's check the shape (rows, columns) of each file so we know what we're working with.

In [8]:
print("fact_log:", fact_log.shape)
print("fact_daily:", fact_daily.shape)
print("dim_developer:", dim_developer.shape)
print("dim_activity:", dim_activity.shape)
print("dim_interruption:", dim_interruption.shape)
print("dim_date:", dim_date.shape)

fact_log: (158152, 11)
fact_daily: (9600, 10)
dim_developer: (150, 6)
dim_activity: (6, 4)
dim_interruption: (8, 6)
dim_date: (90, 10)


## Step 3: Clean `fact_developer_activity_log`




In [9]:
fact_log.head()

,log_id,developer_id,date_key,timestamp_start,timestamp_end,activity_id,interruption_id,session_duration_minutes,in_flow_state,context_switch_flag,cognitive_recovery_minutes
0,1000001,101,20260101,2026-01-01 09:00:00,2026-01-01 09:39:00,1,6,39,True,1,7.24
1,1000002,101,20260101,2026-01-01 09:39:00,2026-01-01 10:06:00,1,4,27,True,1,30.40
2,1000003,101,20260101,2026-01-01 10:06:00,2026-01-01 10:44:00,1,0,38,True,0,0.00
3,1000004,101,20260101,2026-01-01 10:44:00,2026-01-01 10:59:00,6,0,15,False,0,0.00
4,1000005,101,20260101,2026-01-01 10:59:00,2026-01-01 11:14:00,3,0,15,False,0,0.00


Check for duplicate rows and missing values first.

In [10]:
print("Duplicate rows:", fact_log.duplicated().sum())
print("Missing values per column:")
print(fact_log.isnull().sum())

Duplicate rows: 0
Missing values per column:
log_id                        0
developer_id                  0
date_key                      0
timestamp_start               0
timestamp_end                 0
activity_id                   0
interruption_id               0
session_duration_minutes      0
in_flow_state                 0
context_switch_flag           0
cognitive_recovery_minutes    0
dtype: int64


 no duplicates and no missing values. Now let's fix the timestamp columns, since right now they're just text, not real dates.

In [11]:
fact_log["timestamp_start"] = pd.to_datetime(fact_log["timestamp_start"])
fact_log["timestamp_end"] = pd.to_datetime(fact_log["timestamp_end"])
fact_log.dtypes

log_id                                 int64
developer_id                           int64
date_key                               int64
timestamp_start               datetime64[us]
timestamp_end                 datetime64[us]
activity_id                            int64
interruption_id                        int64
session_duration_minutes               int64
in_flow_state                           bool
context_switch_flag                    int64
cognitive_recovery_minutes           float64
dtype: object

Quick sanity check  make sure no session ends before it starts, and no negative durations.

In [12]:
print("Sessions ending before they start:", (fact_log["timestamp_end"] < fact_log["timestamp_start"]).sum())
print("Negative session durations:", (fact_log["session_duration_minutes"] < 0).sum())

Sessions ending before they start: 0
Negative session durations: 0


Sort the data by developer and then by time, so each person's sessions are in order.

In [13]:
fact_log = fact_log.sort_values(["developer_id", "timestamp_start"])
fact_log = fact_log.reset_index(drop=True)
fact_log.head()

,log_id,developer_id,date_key,timestamp_start,timestamp_end,activity_id,interruption_id,session_duration_minutes,in_flow_state,context_switch_flag,cognitive_recovery_minutes
0,1000001,101,20260101,2026-01-01 09:00:00,2026-01-01 09:39:00,1,6,39,True,1,7.24
1,1000002,101,20260101,2026-01-01 09:39:00,2026-01-01 10:06:00,1,4,27,True,1,30.40
2,1000003,101,20260101,2026-01-01 10:06:00,2026-01-01 10:44:00,1,0,38,True,0,0.00
3,1000004,101,20260101,2026-01-01 10:44:00,2026-01-01 10:59:00,6,0,15,False,0,0.00
4,1000005,101,20260101,2026-01-01 10:59:00,2026-01-01 11:14:00,3,0,15,False,0,0.00


## Step 4: Clean `fact_flow_daily`
 

In [14]:
fact_daily.head()

,developer_id,date_key,total_tracked_minutes,deep_work_minutes,pure_flow_minutes,total_interruptions,ci_cd_interruptions,total_cognitive_tax_minutes,flow_efficiency_pct,cognitive_friction_score
0,101,20260101,510,349,326,4,0,67.74,82.80,56.45
1,101,20260102,510,395,365,5,2,111.55,76.59,92.96
2,101,20260105,510,455,374,5,0,70.70,84.10,58.92
3,101,20260106,510,375,275,4,1,83.96,76.61,69.97
4,101,20260107,510,390,377,2,1,34.89,91.53,29.08


In [15]:
print("Duplicate rows:", fact_daily.duplicated().sum())
print("Missing values:")
print(fact_daily.isnull().sum())

Duplicate rows: 0
Missing values:
developer_id                   0
date_key                       0
total_tracked_minutes          0
deep_work_minutes              0
pure_flow_minutes              0
total_interruptions            0
ci_cd_interruptions            0
total_cognitive_tax_minutes    0
flow_efficiency_pct            0
cognitive_friction_score       0
dtype: int64


Check that the numbers make sense deep work minutes shouldn't be more than total tracked minutes, and nothing should be negative.

In [16]:
print("deep_work_minutes bigger than total_tracked_minutes:",
      (fact_daily["deep_work_minutes"] > fact_daily["total_tracked_minutes"]).sum())
print("Any negative numbers in the numeric columns:",
      (fact_daily.select_dtypes("number") < 0).sum().sum())

deep_work_minutes bigger than total_tracked_minutes: 0
Any negative numbers in the numeric columns: 0


In [17]:
fact_daily = fact_daily.sort_values(["developer_id", "date_key"])
fact_daily = fact_daily.reset_index(drop=True)
fact_daily.head()

,developer_id,date_key,total_tracked_minutes,deep_work_minutes,pure_flow_minutes,total_interruptions,ci_cd_interruptions,total_cognitive_tax_minutes,flow_efficiency_pct,cognitive_friction_score
0,101,20260101,510,349,326,4,0,67.74,82.80,56.45
1,101,20260102,510,395,365,5,2,111.55,76.59,92.96
2,101,20260105,510,455,374,5,0,70.70,84.10,58.92
3,101,20260106,510,375,275,4,1,83.96,76.61,69.97
4,101,20260107,510,390,377,2,1,34.89,91.53,29.08


## Step 5: Clean `dim_developer`

 
 

In [18]:
dim_developer.head()

,developer_id,developer_name,team_name,seniority_level,primary_ide,timezone
0,101,Marcus Vance 1,Infra & CI/CD,Senior,VS Code,UTC-8 (PST)
1,102,Nia Campbell 2,Frontend Platform,Senior,VS Code,UTC-5 (EST)
2,103,Fatima Al-Mansoor 3,Core Engine,Mid-Level,IntelliJ IDEA,UTC+5:30 (IST)
3,104,Devon Brooks 4,Core Engine,Mid-Level,IntelliJ IDEA,UTC-5 (EST)
4,105,Elena Rostova 5,Frontend Platform,Junior,VS Code,UTC-5 (EST)


In [19]:
print("Duplicate rows:", dim_developer.duplicated().sum())
print("Duplicate developer_id:", dim_developer["developer_id"].duplicated().sum())

Duplicate rows: 0
Duplicate developer_id: 0


Strip any extra spaces from the text columns, just in case.

In [20]:
dim_developer["developer_name"] = dim_developer["developer_name"].str.strip()
dim_developer["team_name"] = dim_developer["team_name"].str.strip()
dim_developer["seniority_level"] = dim_developer["seniority_level"].str.strip()
dim_developer["primary_ide"] = dim_developer["primary_ide"].str.strip()
dim_developer["timezone"] = dim_developer["timezone"].str.strip()

## Step 6: Clean `dim_activity_type`




In [21]:
dim_activity

,activity_id,activity_name,cognitive_category,is_deep_work
0,1,Active IDE Coding & Refactoring,Deep Work,True
1,2,Local Debugging & Profiling,Deep Work,True
2,3,Code Review & PR Inspection,Focused Task,False
3,4,Technical Documentation / RFC,Focused Task,False
4,5,Scheduled Scrum / Sprint Sync,Collaborative / Meeting,False
5,6,Triaging Slack & Notifications,Friction / Interruption,False


In [22]:
dim_activity["activity_name"] = dim_activity["activity_name"].str.strip()
dim_activity["cognitive_category"] = dim_activity["cognitive_category"].str.strip()

## Step 7: Clean `dim_interruption`





In [23]:
dim_interruption

,interruption_id,source_channel,interruption_category,urgency_tier,requires_action,avg_recovery_latency_min
0,0,None (Continuous Focus),NaN,NaN,False,0.0
1,1,Slack - Automated CI/CD Alert,Tooling & Automation,Informational,False,15.5
2,2,Slack - Direct Message,Direct Communication,Medium,True,18.0
3,3,Slack - @here / @channel,Broadcast Noise,Noise/Spam,False,12.0
4,4,PagerDuty Incident Alert,Production Alert,Critical,True,26.5
5,5,Unplanned Ad-Hoc Call / Huddle,Direct Communication,High,True,24.0
6,6,Jira Notification Digest,Tooling & Automation,Noise/Spam,False,8.5
7,7,GitHub PR Review Requested,Peer Collaboration,Medium,True,16.0


In [24]:
dim_interruption.isnull().sum()

interruption_id             0
source_channel              0
interruption_category       1
urgency_tier                1
requires_action             0
avg_recovery_latency_min    0
dtype: int64

row 0 is supposed to say "None" (meaning "no
interruption happened"), but pandas reads that as a blank/missing value
instead of actual text.  

In [25]:
dim_interruption["interruption_category"] = dim_interruption["interruption_category"].fillna("None")
dim_interruption["urgency_tier"] = dim_interruption["urgency_tier"].fillna("None")
dim_interruption.isnull().sum()

interruption_id             0
source_channel              0
interruption_category       0
urgency_tier                0
requires_action             0
avg_recovery_latency_min    0
dtype: int64

## Step 8: Clean `dim_date`




In [26]:
dim_date.head()

,date_key,date,day_of_week,day_number_in_week,is_weekend,week_number,month_name,quarter,year,sprint_name
0,20260101,2026-01-01,Thursday,4,False,1,January,Q1,2026,Sprint 2026-S01
1,20260102,2026-01-02,Friday,5,False,1,January,Q1,2026,Sprint 2026-S01
2,20260103,2026-01-03,Saturday,6,True,1,January,Q1,2026,Sprint 2026-S01
3,20260104,2026-01-04,Sunday,7,True,1,January,Q1,2026,Sprint 2026-S01
4,20260105,2026-01-05,Monday,1,False,2,January,Q1,2026,Sprint 2026-S01


In [27]:
dim_date["date"] = pd.to_datetime(dim_date["date"])
dim_date.dtypes

date_key                       int64
date                  datetime64[us]
day_of_week                      str
day_number_in_week             int64
is_weekend                      bool
week_number                    int64
month_name                       str
quarter                          str
year                           int64
sprint_name                      str
dtype: object

In [28]:
print("Duplicate dates:", dim_date["date_key"].duplicated().sum())

Duplicate dates: 0


## Step 9: Make sure all the IDs actually match up between files

Before joining the tables together, let's check that every developer_id,
activity_id, interruption_id, and date_key in the main log actually exists
in its matching lookup table.

In [29]:
print("developer_id in fact_log but missing from dim_developer:",
      (~fact_log["developer_id"].isin(dim_developer["developer_id"])).sum())
print("activity_id in fact_log but missing from dim_activity:",
      (~fact_log["activity_id"].isin(dim_activity["activity_id"])).sum())
print("interruption_id in fact_log but missing from dim_interruption:",
      (~fact_log["interruption_id"].isin(dim_interruption["interruption_id"])).sum())
print("date_key in fact_log but missing from dim_date:",
      (~fact_log["date_key"].isin(dim_date["date_key"])).sum())

developer_id in fact_log but missing from dim_developer: 0
activity_id in fact_log but missing from dim_activity: 0
interruption_id in fact_log but missing from dim_interruption: 0
date_key in fact_log but missing from dim_date: 0


## Step 10: Save the cleaned files

Save each cleaned table separately. The next notebook (`Dev_Flow_Data_Merging.ipynb`)
picks these files back up to join them all together.

In [30]:
fact_log.to_csv("fact_developer_activity_log_cleaned.csv", index=False)
fact_daily.to_csv("fact_flow_daily_cleaned.csv", index=False)
dim_developer.to_csv("dim_developer_cleaned.csv", index=False)
dim_activity.to_csv("dim_activity_type_cleaned.csv", index=False)
dim_interruption.to_csv("dim_interruption_cleaned.csv", index=False)
dim_date.to_csv("dim_date_cleaned.csv", index=False)

print("All cleaned files saved!")

All cleaned files saved!


## Summary

- Loaded all 6 raw files and checked their shapes
- Checked each file for duplicates and missing values — the raw files were already pretty clean
- Fixed the timestamp/date columns so they're proper date types, not just text
- Checked that the numbers made sense (no negative values, no impossible combinations)
- Found and fixed one tricky issue: the word "None" in `dim_interruption.csv` was being read as a missing value instead of actual text
- Checked that every ID in the main log matches up correctly with the lookup tables
- Saved all 6 cleaned files, ready to be merged together in the next notebook
